In [ ]:
# ============================================================
# 环境配置
# - Colab 用户：取消注释下方 Colab 区块
# - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch matplotlib numpy -q

# ── 本地 Jupyter 环境（仅缺包时安装） ──
import importlib
import subprocess
import sys

def _ensure_package(pkg):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["torch", "matplotlib", "numpy"]:
    _ensure_package(pkg)

In [ ]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# SlowFast 代码实战：学习实现 vs 工程实现

基于论文 *SlowFast Networks for Video Recognition*（Feichtenhofer et al., ICCV 2019），
用 **toy 视频动作分类** 任务演示 SlowFast 的核心架构。

本 Notebook 包含两条并行路径，使用 **相同的任务和数据**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解双路径时空建模 | 掌握更紧凑的工程实现 |
| 实现方式 | 手写 Slow / Fast / Lateral / Fusion | 用更少代码封装同一思路 |
| 代码量 | ~120 行核心模型代码 | ~60 行核心模型代码 |
| 适合场景 | 面试准备、原理拆解 | 原型验证、工程表达 |

> 两条路径不是两套无关代码，而是同一 SlowFast 思想在不同抽象层级上的表达。

## 一、论文背景、任务与范围

**论文**：*SlowFast Networks for Video Recognition*  
**作者**：Christoph Feichtenhofer, Haoqi Fan, Jitendra Malik, Kaiming He  
**会议**：ICCV 2019

### 1. SlowFast 想解决什么问题？
传统 3D CNN 往往把“看清楚空间语义”和“看清楚快速运动”交给同一条路径，结果要么时间分辨率不够，要么计算量太大。  
SlowFast 的核心思想是 **分工**：

- **Slow pathway**：低帧率、较强通道容量，负责“看清楚是什么”
- **Fast pathway**：高帧率、较少通道，负责“看清楚怎么动”
- **Lateral connection**：把 Fast 捕捉到的动态信息注入 Slow

### 2. 为什么这个 toy 任务足够说明问题？
本 notebook 使用合成视频动作分类：方块在 16 帧里做不同方向的移动或扩张。  
它虽然不是 Kinetics，但已经包含 SlowFast 最关心的两件事：

- **空间语义**：目标形状与位置
- **时间动态**：目标如何随时间变化

### 3. 架构谱系与边界
可以把它放在这条线上理解：

`Two-Stream → I3D / R(2+1)D → SlowFast → Video Transformer`

本 notebook **覆盖**：
- Slow / Fast 双路径采样
- Lateral fusion
- 从零实现与更简洁工程实现
- 训练、推理、对比、面试表达

本 notebook **不覆盖**：
- Kinetics 级大规模训练
- 预训练 SlowFast-R50 推理部署
- 检测任务（AVA）

## 二、最小必要理论

完整理论见配套 `14_SlowFast.md`；这里只保留理解代码所需的最小公式。

### 1. 双路径采样
给定原始视频 `x ∈ R^{B×C×T×H×W}`，SlowFast 先做两种时间采样：

$$x_{slow} = Sample_{\tau}(x), \qquad x_{fast} = Sample_{\tau / \alpha}(x)$$

- `τ`：Slow 路径的时间步长
- `α`：Fast 相对 Slow 的速度比

因此：
- Slow 看到更少帧，但更适合提取稳定语义
- Fast 看到更多帧，但通道更少，重点是运动变化

### 2. 横向连接
Fast 路径的特征经过时间对齐后注入 Slow 路径：

$$\tilde{F}_{fast} = Conv_t(F_{fast})$$
$$F_{slow}^{new} = F_{slow}^{old} + \tilde{F}_{fast}$$

本 notebook 采用 **time-strided Conv3d + add fusion**，因为它既能对齐时间维度，又能保持代码与理论说明一致。

### 3. 最终融合
两条路径在尾部各自做全局平均池化，然后拼接分类：

$$z = [GAP(F_{slow}),\ GAP(F_{fast})]$$
$$y = FC(z)$$

### 4. 本 notebook 的教学取值
为了适配 CPU / Colab 免费环境，我们把论文里的大模型缩小为：

- `T = 16`
- `τ = 4`
- `α = 4`
- `β = 1 / 4`（论文常用更小 Fast 通道占比，这里适当放宽以保证小模型稳定训练）

## 三、组件映射表

| 论文组件 | 学习路径覆盖 | 工程库 / API 对应 | 状态 |
|---|---|---|---|
| Slow pathway | `slow_stem` + `slow_block2/3` 完整实现 | `pytorchvideo.models.slowfast.create_slowfast` 中的 slow pathway | Runnable |
| Fast pathway | `fast_stem` + `fast_block2/3` 完整实现 | `create_slowfast` 中的 fast pathway | Runnable |
| 双路径采样 `(τ, α)` | `sample_frames()` 显式实现 | PyTorchVideo 里常见的 `PackPathway()` / 手动 pathway packing | Runnable |
| Lateral connection | `LateralConnection` 显式实现 | SlowFast fusion 模块 | Runnable |
| Final fusion | `GAP + concat + FC` | 分类 head | Runnable |
| 轻量工程表达 | `ConciseSlowFast` | 更高层生态可切换到 `torch.hub.load(..., 'slowfast_r50')` | Illustrative |
| 大规模预训练 SlowFast | 本 notebook 只解释，不直接运行 | TorchHub / PyTorchVideo 预训练模型 | Explain-only |

## 四、数据准备

真实视频数据集如 Kinetics 体量太大，不适合拿来做一份可即开即跑的教学 notebook。  
这里构造一个 **synthetic motion dataset**：

- 输入形状固定为 `(3, 16, 32, 32)`
- 类别包括：`left / right / up / down / expand`
- 每段视频里都有一个亮方块，按不同模式运动
- 加入轻微噪声，避免任务过于死板

这个 toy 任务虽然简单，但它已经足以让 Slow 路径学习“稳定语义”，让 Fast 路径学习“细粒度运动”。

In [ ]:
class SyntheticMotionVideoDataset(Dataset):
    action_names = ["left", "right", "up", "down", "expand"]

    def __init__(self, num_samples, num_frames=16, img_size=32, square_size=6, seed=42):
        super().__init__()
        self.num_samples = num_samples
        self.num_frames = num_frames
        self.img_size = img_size
        self.square_size = square_size
        self.rng = np.random.RandomState(seed)
        self.videos = []
        self.labels = []

        for i in range(num_samples):
            label = i % len(self.action_names)
            self.videos.append(self._make_video(label))
            self.labels.append(label)

    def _make_video(self, label):
        T = self.num_frames
        H = W = self.img_size
        s = self.square_size
        max_x = W - s - 1
        max_y = H - s - 1

        if label == 0:      # left
            x0 = self.rng.randint(T - 1, max_x + 1)
            y0 = self.rng.randint(4, max_y - 3)
            dx, dy = -1, 0
        elif label == 1:    # right
            x0 = self.rng.randint(0, max_x - T + 2)
            y0 = self.rng.randint(4, max_y - 3)
            dx, dy = 1, 0
        elif label == 2:    # up
            x0 = self.rng.randint(4, max_x - 3)
            y0 = self.rng.randint(T - 1, max_y + 1)
            dx, dy = 0, -1
        elif label == 3:    # down
            x0 = self.rng.randint(4, max_x - 3)
            y0 = self.rng.randint(0, max_y - T + 2)
            dx, dy = 0, 1
        else:               # expand
            x0 = W // 2 - s // 2
            y0 = H // 2 - s // 2
            dx, dy = 0, 0

        frames = []
        for t in range(T):
            canvas = np.zeros((H, W), dtype=np.float32)
            if label < 4:
                x = x0 + dx * t
                y = y0 + dy * t
                canvas[y:y + s, x:x + s] = 1.0
            else:
                grow = min(1 + t // 4, 4)
                x1 = max(0, x0 - grow)
                y1 = max(0, y0 - grow)
                x2 = min(W, x0 + s + grow)
                y2 = min(H, y0 + s + grow)
                canvas[y1:y2, x1:x2] = 1.0

            canvas += self.rng.normal(0.0, 0.03, size=canvas.shape).astype(np.float32)
            canvas = np.clip(canvas, 0.0, 1.0)
            frame = torch.from_numpy(canvas).unsqueeze(0).repeat(3, 1, 1)  # (3, H, W)
            frames.append(frame)

        return torch.stack(frames, dim=1)  # (3, T, H, W)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], int(self.labels[idx])

In [ ]:
PREVIEW_BATCH_SIZE = 16

train_ds = SyntheticMotionVideoDataset(num_samples=240, seed=42)
test_ds = SyntheticMotionVideoDataset(num_samples=80, seed=123)
ACTION_NAMES = train_ds.action_names

preview_loader = DataLoader(train_ds, batch_size=PREVIEW_BATCH_SIZE, shuffle=True)

x, y = next(iter(preview_loader))
print(f"Video batch: {tuple(x.shape)}")
print(f"Label batch: {tuple(y.shape)}")
print(f"Classes: {ACTION_NAMES}")

sample_video = x[0]
sample_label = ACTION_NAMES[y[0].item()]
frame_ids = [0, 4, 8, 12, 15]

fig, axes = plt.subplots(1, len(frame_ids), figsize=(12, 2.6))
for ax, frame_id in zip(axes, frame_ids):
    ax.imshow(sample_video[:, frame_id].permute(1, 2, 0).numpy())
    ax.set_title(f"Frame {frame_id}")
    ax.axis("off")

fig.suptitle(f"Toy action: {sample_label}")
plt.tight_layout()
plt.show()

## 五、共享组件

先把两条路径共用的超参数、采样函数、训练/评估逻辑集中起来。这样后面无论是学习路径还是工程路径，都能保证：

- 任务相同
- 数据相同
- 训练轮数与优化器设置相同
- 对比结果更公平

In [ ]:
# ── 超参数（两条路径共用）──
TAU = 4
ALPHA = 4
BETA_INV = 4
BASE_CH = 16
LR = 2e-3
NUM_EPOCHS = 6
BATCH_SIZE = 16
NUM_CLASSES = len(ACTION_NAMES)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)


def sample_frames(videos, tau, alpha):
    """原始视频 -> Slow / Fast 两条路径输入
    videos: (B, C, T, H, W)
    """
    assert tau % alpha == 0, "This notebook assumes tau is divisible by alpha."
    t_raw = videos.size(2)
    assert t_raw % tau == 0, "This notebook assumes clip length is divisible by tau."

    slow_idx = torch.arange(0, t_raw, tau)
    fast_stride = tau // alpha
    fast_idx = torch.arange(0, t_raw, fast_stride)

    x_slow = videos.index_select(2, slow_idx.to(videos.device))          # (B, C, T_slow, H, W)
    x_fast = videos.index_select(2, fast_idx.to(videos.device))          # (B, C, T_fast, H, W)
    assert x_fast.size(2) == x_slow.size(2) * alpha, "Slow/Fast temporal alignment mismatch."
    return x_slow, x_fast


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def train_model(model, train_loader, num_epochs, lr, device, tau, alpha, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    history = {"loss": [], "acc": []}

    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for videos, labels in train_loader:
            videos = videos.to(device)
            labels = labels.to(device)
            x_slow, x_fast = sample_frames(videos, tau=tau, alpha=alpha)
            logits = model(x_slow, x_fast)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_acc = correct / total
        history["loss"].append(epoch_loss)
        history["acc"].append(epoch_acc)
        print(f"Epoch {epoch + 1:02d}/{num_epochs} | loss={epoch_loss:.4f} | acc={epoch_acc:.2%}")

    return history


@torch.no_grad()
def evaluate_model(model, data_loader, device, tau, alpha):
    model = model.to(device)
    model.eval()
    correct, total = 0, 0

    for videos, labels in data_loader:
        videos = videos.to(device)
        labels = labels.to(device)
        x_slow, x_fast = sample_frames(videos, tau=tau, alpha=alpha)
        logits = model(x_slow, x_fast)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total


@torch.no_grad()
def predict_batch(model, videos, device, tau, alpha):
    model = model.to(device)
    model.eval()
    videos = videos.to(device)
    x_slow, x_fast = sample_frames(videos, tau=tau, alpha=alpha)
    logits = model(x_slow, x_fast)
    return logits.cpu(), logits.argmax(dim=1).cpu()


x_slow, x_fast = sample_frames(x, tau=TAU, alpha=ALPHA)
print(f"Slow input shape: {tuple(x_slow.shape)}")
print(f"Fast input shape: {tuple(x_fast.shape)}")

## 六、学习路径：从零实现 SlowFast

本节选择 **L1: full mini training**。  
也就是说，我们不只验证单个组件的 shape，还会真正训练一个小型 SlowFast 分类器。

实现顺序遵循论文数据流：

`原始视频 -> 双路径采样 -> stem -> lateral fusion -> block -> final fusion -> classifier`

### 1. 基础 3D 残差块

Slow / Fast 两条路径都建立在 3D 卷积之上。这里用一个最小化的残差块承载时空特征提取：

$$y = ReLU(BN(Conv3d(x)) + shortcut(x))$$

输入输出遵循 PyTorch 3D 模块的标准格式：

- 输入：`(B, C_in, T, H, W)`
- 输出：`(B, C_out, T', H', W')`

其中时间维通常保持不变，空间维可以通过 stride 下采样。

In [ ]:
class BasicBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, spatial_stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=(1, spatial_stride, spatial_stride), padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_ch)
        self.relu = nn.ReLU(inplace=True)

        if in_ch != out_ch or spatial_stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, kernel_size=1, stride=(1, spatial_stride, spatial_stride), bias=False),
                nn.BatchNorm3d(out_ch),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)                                       # (B, C_out, T, H', W')
        out = self.relu(self.bn1(self.conv1(x)))                         # (B, C_out, T, H', W')
        out = self.bn2(self.conv2(out))                                  # (B, C_out, T, H', W')
        out = self.relu(out + identity)                                  # (B, C_out, T, H', W')
        return out


block = BasicBlock3D(3, 16, spatial_stride=2)
print(f"BasicBlock3D output: {tuple(block(torch.randn(2, 3, 4, 32, 32)).shape)}")

### 2. 横向连接：Fast -> Slow

SlowFast 的关键不是简单地并联两条路径，而是让 Fast 路径把运动信息注入 Slow 路径。  
这里采用最直接的 **时间步长卷积 + 残差相加**：

$$\tilde{F}_{fast} = Conv3d(F_{fast}, kernel=(\alpha, 1, 1), stride=(\alpha, 1, 1))$$
$$F_{slow}^{new} = F_{slow}^{old} + \tilde{F}_{fast}$$

shape 变化：

- `x_fast`: `(B, C_fast, T_fast, H, W)`
- `aligned_fast`: `(B, C_slow, T_slow, H, W)`
- `out`: `(B, C_slow, T_slow, H, W)`

In [ ]:
class LateralConnection(nn.Module):
    def __init__(self, fast_ch, slow_ch, alpha):
        super().__init__()
        self.proj = nn.Conv3d(
            fast_ch,
            slow_ch,
            kernel_size=(alpha, 1, 1),
            stride=(alpha, 1, 1),
            bias=False,
        )
        self.bn = nn.BatchNorm3d(slow_ch)

    def forward(self, x_slow, x_fast):
        aligned_fast = self.bn(self.proj(x_fast))                        # (B, C_slow, T_slow, H, W)
        assert aligned_fast.shape == x_slow.shape, "Lateral fusion shape mismatch."
        return x_slow + aligned_fast                                     # (B, C_slow, T_slow, H, W)


lat = LateralConnection(fast_ch=4, slow_ch=16, alpha=4)
s = torch.randn(2, 16, 4, 16, 16)
f = torch.randn(2, 4, 16, 16, 16)
print(f"Lateral output: {tuple(lat(s, f).shape)}")

### 3. Slow / Fast 两条路径如何分工？

**Slow pathway**：
- 输入帧少：`T_slow = T_raw / τ`
- 通道较多：负责稳定语义
- 第一层时间核通常较小，强调空间结构

**Fast pathway**：
- 输入帧多：`T_fast = α * T_slow`
- 通道较少：负责运动变化
- 第一层时间核更大，强调早期时间建模

本 notebook 的一个最小 stage 设计如下：

| 路径 | Stage 1 | Stage 2 | Stage 3 |
|---|---|---|---|
| Slow | stem -> `BASE_CH` | block -> `2*BASE_CH` | block -> `4*BASE_CH` |
| Fast | stem -> `BASE_CH/BETA_INV` | block -> `2*...` | block -> `4*...` |

### 4. 组装完整 SlowFast 模型

下面把双路径、横向连接、尾部融合组装成完整分类器。

最终数据流：

1. 原始视频 -> `sample_frames()`
2. Slow / Fast 分别过 stem
3. Fast 通过 lateral 对齐后注入 Slow
4. 两条路径各自再过一个 block
5. 分别 `GAP`
6. `concat`
7. `FC` 输出类别

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


class SlowFastNet(nn.Module):
    def __init__(self, num_classes=5, base_ch=16, alpha=4, beta_inv=4):
        super().__init__()
        fast_ch = base_ch // beta_inv

        # Slow pathway
        self.slow_stem = nn.Sequential(
            nn.Conv3d(3, base_ch, kernel_size=(1, 5, 5), stride=(1, 2, 2), padding=(0, 2, 2), bias=False),
            nn.BatchNorm3d(base_ch),
            nn.ReLU(inplace=True),
        )
        self.slow_block2 = BasicBlock3D(base_ch, base_ch * 2, spatial_stride=2)
        self.slow_block3 = BasicBlock3D(base_ch * 2, base_ch * 4, spatial_stride=2)

        # Fast pathway
        self.fast_stem = nn.Sequential(
            nn.Conv3d(3, fast_ch, kernel_size=(5, 5, 5), stride=(1, 2, 2), padding=(2, 2, 2), bias=False),
            nn.BatchNorm3d(fast_ch),
            nn.ReLU(inplace=True),
        )
        self.fast_block2 = BasicBlock3D(fast_ch, fast_ch * 2, spatial_stride=2)
        self.fast_block3 = BasicBlock3D(fast_ch * 2, fast_ch * 4, spatial_stride=2)

        # Lateral
        self.lateral1 = LateralConnection(fast_ch, base_ch, alpha)
        self.lateral2 = LateralConnection(fast_ch * 2, base_ch * 2, alpha)

        # Head
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(base_ch * 4 + fast_ch * 4, num_classes)

    def forward(self, x_slow, x_fast):
        s1 = self.slow_stem(x_slow)                                          # (B, base, T_s, 16, 16)
        f1 = self.fast_stem(x_fast)                                          # (B, fast, T_f, 16, 16)

        s1 = self.lateral1(s1, f1)                                           # (B, base, T_s, 16, 16)
        s2 = self.slow_block2(s1)                                            # (B, 2*base, T_s, 8, 8)
        f2 = self.fast_block2(f1)                                            # (B, 2*fast, T_f, 8, 8)

        s2 = self.lateral2(s2, f2)                                           # (B, 2*base, T_s, 8, 8)
        s3 = self.slow_block3(s2)                                            # (B, 4*base, T_s, 4, 4)
        f3 = self.fast_block3(f2)                                            # (B, 4*fast, T_f, 4, 4)

        s_out = self.pool(s3).flatten(1)                                     # (B, 4*base)
        f_out = self.pool(f3).flatten(1)                                     # (B, 4*fast)
        out = torch.cat([s_out, f_out], dim=1)                               # (B, 4*base + 4*fast)
        return self.fc(out)                                                  # (B, num_classes)


model_learning = SlowFastNet(
    num_classes=NUM_CLASSES,
    base_ch=BASE_CH,
    alpha=ALPHA,
    beta_inv=BETA_INV,
)

xs, xf = sample_frames(torch.randn(2, 3, 16, 32, 32), tau=TAU, alpha=ALPHA)
out = model_learning(xs, xf)
print(f"Learning model output shape: {tuple(out.shape)}")
print(f"Learning model params: {count_parameters(model_learning):,}")

In [ ]:
print("=" * 60)
print("Learning path: training SlowFastNet")
print("=" * 60)

learning_history = train_model(
    model_learning,
    train_loader,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    device=device,
    tau=TAU,
    alpha=ALPHA,
    seed=42,
)
learning_acc = evaluate_model(model_learning, test_loader, device=device, tau=TAU, alpha=ALPHA)
print(f"Learning path test acc: {learning_acc:.2%}")

## 七、工程路径：更紧凑的 PyTorch 写法

这里的工程路径不是直接调用重型预训练 SlowFast-R50，而是保留同一任务与同一数据，改用更紧凑的写法表达相同架构思想。

这样做的原因：

- `torchvision` 当前没有原生 SlowFast builder
- `pytorchvideo` 虽然提供 `create_slowfast` / TorchHub `slowfast_r50`，但额外依赖与模型体量不适合这份轻量教学 notebook
- 我们更关心“如何把论文结构写成更工程化的 PyTorch 代码”

### 对应关系

| 学习路径实现 | 工程路径实现 | 更高层生态 |
|---|---|---|
| `BasicBlock3D` | 更少中间变量的 block 组合 | PyTorchVideo internals |
| `LateralConnection` | 直接内联投影与相加 | SlowFast fusion module |
| `SlowFastNet` | `ConciseSlowFast` | `create_slowfast()` |
| 手动 `sample_frames()` | 同样复用 | `PackPathway()` 思路 |

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


class ConciseSlowFast(nn.Module):
    def __init__(self, num_classes=5, base_ch=16, alpha=4, beta_inv=4):
        super().__init__()
        fast_ch = base_ch // beta_inv

        self.s_stem = nn.Sequential(
            nn.Conv3d(3, base_ch, kernel_size=(1, 5, 5), stride=(1, 2, 2), padding=(0, 2, 2), bias=False),
            nn.BatchNorm3d(base_ch),
            nn.ReLU(inplace=True),
        )
        self.f_stem = nn.Sequential(
            nn.Conv3d(3, fast_ch, kernel_size=(5, 5, 5), stride=(1, 2, 2), padding=(2, 2, 2), bias=False),
            nn.BatchNorm3d(fast_ch),
            nn.ReLU(inplace=True),
        )

        self.s_b2 = BasicBlock3D(base_ch, base_ch * 2, spatial_stride=2)
        self.s_b3 = BasicBlock3D(base_ch * 2, base_ch * 4, spatial_stride=2)
        self.f_b2 = BasicBlock3D(fast_ch, fast_ch * 2, spatial_stride=2)
        self.f_b3 = BasicBlock3D(fast_ch * 2, fast_ch * 4, spatial_stride=2)

        self.lat1_proj = nn.Conv3d(fast_ch, base_ch, kernel_size=(alpha, 1, 1), stride=(alpha, 1, 1), bias=False)
        self.lat1_bn = nn.BatchNorm3d(base_ch)
        self.lat2_proj = nn.Conv3d(fast_ch * 2, base_ch * 2, kernel_size=(alpha, 1, 1), stride=(alpha, 1, 1), bias=False)
        self.lat2_bn = nn.BatchNorm3d(base_ch * 2)

        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(base_ch * 4 + fast_ch * 4, num_classes)

    def forward(self, x_slow, x_fast):
        s = self.s_stem(x_slow)
        f = self.f_stem(x_fast)

        s = s + self.lat1_bn(self.lat1_proj(f))
        s = self.s_b2(s)
        f = self.f_b2(f)

        s = s + self.lat2_bn(self.lat2_proj(f))
        s = self.s_b3(s)
        f = self.f_b3(f)

        s = self.pool(s).flatten(1)
        f = self.pool(f).flatten(1)
        return self.fc(torch.cat([s, f], dim=1))


model_engineering = ConciseSlowFast(
    num_classes=NUM_CLASSES,
    base_ch=BASE_CH,
    alpha=ALPHA,
    beta_inv=BETA_INV,
)

out = model_engineering(xs, xf)
print(f"Engineering model output shape: {tuple(out.shape)}")
print(f"Engineering model params: {count_parameters(model_engineering):,}")

In [ ]:
print("=" * 60)
print("Engineering path: training ConciseSlowFast")
print("=" * 60)

engineering_history = train_model(
    model_engineering,
    train_loader,
    num_epochs=NUM_EPOCHS,
    lr=LR,
    device=device,
    tau=TAU,
    alpha=ALPHA,
    seed=42,
)
engineering_acc = evaluate_model(model_engineering, test_loader, device=device, tau=TAU, alpha=ALPHA)
print(f"Engineering path test acc: {engineering_acc:.2%}")

## 八、学习路径 vs 工程路径对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 可以看清 Slow / Fast / lateral 的具体 shape 流动 | 更适合展示如何把论文结构压缩成可复用代码 |
| 代码量 | 约 120 行核心实现 | 约 60 行核心实现 |
| 灵活性 | 高，可单独改每个组件 | 中，结构更紧凑但可读性略低 |
| 稳定性 | 中，手写细节多更容易出错 | 更高，模块边界更少 |
| 工业适配度 | 适合教学、研究原型、面试讲解 | 更接近实际工程里“先写清楚再迭代”的风格 |
| 适用场景 | 原理学习、结构分析 | baseline、轻量实验、论文结构落地 |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(learning_history["loss"], marker="o", label="Learning path")
axes[0].plot(engineering_history["loss"], marker="s", label="Engineering path")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].bar(["Learning", "Engineering"], [learning_acc, engineering_acc], color=["steelblue", "coral"])
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Test Accuracy")
axes[1].set_ylim(0, 1)
for i, v in enumerate([learning_acc, engineering_acc]):
    axes[1].text(i, v + 0.02, f"{v:.2%}", ha="center")

plt.tight_layout()
plt.show()

print(f"Learning path acc   : {learning_acc:.2%}")
print(f"Engineering path acc: {engineering_acc:.2%}")

## 九、训练阶段 vs 推理阶段

SlowFast 不像 Transformer 那样存在 teacher forcing vs autoregressive 这种结构性分叉。  
它在训练和推理阶段的核心前向结构是一样的，差异主要体现在：

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `model.train()`，保留梯度，更新参数 | 相同 |
| 推理 | `model.eval()` + `torch.no_grad()` | 相同 |
| clip 处理 | 同样先做 Slow/Fast 采样 | 相同 |

也就是说，SlowFast 的两条路径差异更多体现在**架构分工**，而不是 train/infer 算法切换。

In [ ]:
demo_videos, demo_labels = next(iter(test_loader))
logits_l, preds_l = predict_batch(model_learning, demo_videos[:6], device=device, tau=TAU, alpha=ALPHA)
logits_e, preds_e = predict_batch(model_engineering, demo_videos[:6], device=device, tau=TAU, alpha=ALPHA)

print("Ground truth :", [ACTION_NAMES[i] for i in demo_labels[:6].tolist()])
print("Learning pred:", [ACTION_NAMES[i] for i in preds_l.tolist()])
print("Engineering :", [ACTION_NAMES[i] for i in preds_e.tolist()])

## 十、结果与边界

### 学习路径得到什么？
- 能看清 `sample_frames -> Slow/Fast -> lateral -> fusion` 的完整 shape 流动
- 能回答“为什么要双路径”“为什么 Fast 通道少”“为什么 Fast -> Slow 单向注入”
- 能在面试中把 SlowFast 和 I3D / Two-Stream / R(2+1)D 说清楚

### 工程路径得到什么？
- 能把相同架构思想写成更短、更适合工程复用的 PyTorch 代码
- 能理解“教程里为什么不直接上预训练 SlowFast-R50”
- 能把理论结构平滑过渡到真实工程实现

### 本 notebook 的边界
- 数据是 toy synthetic video，不代表真实视频基准表现
- 模型规模远小于论文配置
- 这里只讲分类，不讲检测或大规模预训练
- 工程路径是轻量版表达，不是完整工业 SlowFast 训练栈

In [ ]:
# 可视化两条路径的通道容量
stages = ["stem", "block2", "block3", "head"]
slow_ch = [BASE_CH, BASE_CH * 2, BASE_CH * 4, BASE_CH * 4]
fast_ch = [BASE_CH // BETA_INV, BASE_CH // BETA_INV * 2, BASE_CH // BETA_INV * 4, BASE_CH // BETA_INV * 4]

x_pos = np.arange(len(stages))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bar1 = ax.bar(x_pos - width / 2, slow_ch, width, label="Slow pathway", color="steelblue")
bar2 = ax.bar(x_pos + width / 2, fast_ch, width, label="Fast pathway", color="coral")

ax.set_xticks(x_pos)
ax.set_xticklabels(stages)
ax.set_ylabel("Channels")
ax.set_title("Channel Capacity by Stage")
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bar in list(bar1) + list(bar2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5, f"{int(bar.get_height())}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 十一、面试与项目表达

### 高频面试题

**Q1：SlowFast 的核心创新是什么？**

不是“单纯把网络做大”，而是把视频理解拆成两件事：
- Slow 负责低频语义
- Fast 负责高频运动
- 两者通过 lateral connection 协同

**Q2：为什么 Fast 路径要高帧率、低通道？**

因为运动信息主要体现在时间变化，不需要很强的通道容量去建模复杂空间语义。  
高帧率保证它能看到细粒度动态，低通道保证计算成本可控。

**Q3：为什么横向连接通常是 Fast -> Slow，而不是双向？**

Fast 的职责是保持对运动变化的敏感；Slow 的职责是承担主要语义建模。  
让 Fast 把运动情报注入 Slow，比双向混合更符合“分工”的设计哲学。

**Q4：SlowFast 和 Two-Stream 的差别是什么？**

- Two-Stream 通常显式依赖光流
- SlowFast 只用 RGB，但通过双时间尺度隐式建模运动
- SlowFast 是端到端双路径，不需要额外光流预处理

**Q5：SlowFast 和 I3D / R(2+1)D 的区别是什么？**

- I3D：统一 3D 卷积时空建模
- R(2+1)D：把 3D 卷积分解为空间 + 时间
- SlowFast：直接把路径拆成快慢两支，在更高层次做时间分工

**Q6：为什么本 notebook 的工程路径不用 torchvision 直接加载 SlowFast？**

因为当前 `torchvision` 没有原生 SlowFast builder。  
更完整的 SlowFast 生态在 `pytorchvideo` / TorchHub，但为了保证 notebook 轻量、同任务可训练，这里选择更紧凑的 PyTorch 版本。

### 面试速答提纲

| 问题 | 一句话回答 |
|---|---|
| SlowFast 在做什么？ | 用双时间尺度路径分别建模语义和运动 |
| Slow 路径负责什么？ | 低帧率、高语义 |
| Fast 路径负责什么？ | 高帧率、强运动 |
| lateral connection 是什么？ | 把 Fast 的动态信息注入 Slow |
| 为什么 Fast 通道少？ | 为了低成本保留高时域分辨率 |
| 为什么不用光流？ | SlowFast 通过 RGB 双路径隐式学习运动 |
| 工程里怎么落地？ | 用更紧凑的双路径 PyTorch 代码或迁移到 PyTorchVideo |

### 项目表达模板

> 我做过一个 SlowFast 教学型 baseline。  
> 学习路径里，我从零实现了双路径采样、3D block、lateral connection 和最终融合，能清楚解释 SlowFast 为什么要把语义和运动分开建模。  
> 工程路径里，我把同一结构压缩成更紧凑的 PyTorch 代码，同时对照官方生态说明了为什么真实项目里通常会切到 PyTorchVideo。  
> 这个项目让我不仅能讲清 SlowFast 的原理，也能把它和 Two-Stream、I3D、R(2+1)D、TimeSformer 放到同一条视频理解演化线上比较。

### 延伸阅读与对比

| 模型 | 核心思想 | 优势 | 局限 |
|---|---|---|---|
| Two-Stream | RGB + 光流双流 | 运动显式 | 光流成本高 |
| I3D | 膨胀 2D -> 3D | 迁移 ImageNet 预训练 | 时空统一建模较重 |
| R(2+1)D | 3D 分解为 2D+1D | 更易优化 | 仍是单路径 |
| SlowFast | 快慢双路径 | 兼顾语义与运动 | 结构更复杂 |
| TimeSformer | 时空注意力 | 长程依赖强 | 算力需求更高 |

### 进阶探索方向

- 接入 `pytorchvideo.create_slowfast()` 做更接近官方实现的实验
- 把 toy 数据换成真实小型视频数据集
- 尝试不同 `τ / α / β` 组合，观察速度与效果权衡
- 与 I3D / R(2+1)D 做同数据集对照实验